<a href="https://colab.research.google.com/github/adrianacalderon/CaribbeanTraining/blob/main/6_HydroSOS_Outlooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **Introduction**


**HydroSOS** is a World Meteorological Organization initiative that aims to strengthen the capacity of National Hydrological and Meteorological Services to manage water resources through the development of hydrological status and outlook products. HydroSOS seeks to enable an assessment of the current state of water resources relative to "expected" or "normal" conditions, as well as hydrological outlooks on how this state may change over the following weeks and months. Within the context of the HydroSOS initiative, a portal showcasing status and future outlooks has been created, which can be accessed at: https://wmohydrosos.ceh.ac.uk/portal.

In this tutorial, we will use a script to generate **HydroSOS-based hydrological outlooks** at points where streamflow forecasts are available. The methodology consists of transforming the forecasted flows into a percentile relative to the historical record for the corresponding time of year — for example, the current month — and classifying the resulting percentile according to a predetermined flow category.

**Note: Make sure you are using the most up-to-date version of the HydroSOS notebook, available on the HydroSOS GitHub repository.**

---

# **Clone the HydroSOS GitHub repository**

Clone the WMO HydroSOS repository hosted on Github to access the percentile calculation scripts, as well as an example of streamflow forecast results.

*   Note: To use the `git clone` command on another notebook/Python script execution platform (e.g. ***Jupyter***, ***Visual Studio Code***), you need to make sure that `Git` is installed on your computer. If not, git can be downloaded from (https://git-scm.com/downloads)

In [ ]:
!git clone https://github.com/wmo-im/HydroSOS/

---

# **Import Python libraries**

Import the required libraries to run this tutorial:

In [ ]:
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

Create an output folder in the root directory to export the results:

In [ ]:
# Create the folder if it doesn't exist
os.makedirs('output_hydrosos', exist_ok=True)

---
# **1. Apply the HydroSOS methods with the `forecastcalc.py` script**

1.1. The section shows how to generate hydrological outlooks using the  `forecastcalc.py`, script, available in the HydroSOS Github repository (https://github.com/wmo-im/HydroSOS).

* The script will calculate the same categories as `statuscalc.py`, based on historical streamflow data from the reference period (eg. 1991-2020). Note: **if the forecasts to be classified have been generated by a conceptual rainfall-runoff hydrological model, the historical data used here should be the modelled data**, i.e., obtained from a historical simulation with the same model. This reduces the infleunce of model errors (in relation to the mean and the variability of the series)on the calcualtion of percentiles, improving statistical consistency between the future and historical periods.

* Each forecast or ensemble member will have its percentile calculated relative to the observed or simulated historical data series, according to the forecasting methodology adopted.

1.2. The script will process the observed / forecast data pairs **for each available station**, and must be run with the following parameters:

`python forecastcalc.py obs_dir forecast_dir output_dir --obsDirStartingMonth --varName`


Where
* `obs_dir`: directory containing .csv files of historical streamflow data (observed or simulated), as **daily time series**. File names must follow the format **X_CatchmentID.csv**, where "X" is a prefix and "CatchmentID" is the hydrometric station code. See the files in the HydroSOS/example_data/forecast/input/obs_dir/ folder for the expected format of these files;


* `forecast_dir`: directory containing only .csv files of **forecasts with monthly time step**, where each ensemble member must be in a separate .csv file. File names must follow the format **X_ENS_CatchmentID.csv**, where "ENS" is a reference to the member (e.g. A, B, C, ... or M1, M2, M3, etc.). See the files in the HydroSOS/example_data/forecast/input/forecast_dir/ folder for the expected format of these files;

* `output_dir`: directory to save the files processed;

* `obsDirStartingMonth`starting month of the data in the `obs_dir` folder (default January = 1).

* `varName`: Name of the variable in the data (default = Discharge)


**Important**: The reference period will be determined by the availability of historical daily streamflow data. For example, if the data series in the `obs_dir` folder starts on 01/01/1991 and ends on 31/12/2020, the period used to calculate the percentile bands for forecast classification will be 1991-2020.


For more details, see the information available in `HydroSOS/README.md`.

In [ ]:
# Run the script using shell command
!python ./HydroSOS/forecast/forecastcalc.py ./HydroSOS/example_data/forecast/input/obs_dir/ ./HydroSOS/example_data/forecast/input/forecast_dir/ ./output_hydrosos/


---
1.3. If the script runs successfully, several folders containing ".csv" files should be generated in the "output" folder. These folders should include:

* *accumulated*: aggregated forecast product (average of previous months up to the current date)
* *single*: forecast product with monthly time step
* *status*: historical data used to calculate percentiles and generate the categories.

---
# **2. Loading the tables with the HydroSOS results**

2.1. Once the tables have been generated by running the `forecastcalc.py` script, we will set some parameters to read and plot the results:

* `station_id`: hydrometric station code used to read the data;
* `forecast_type`: forecast aggregation to be plotted ("single" or "accumulated");
* `color_codes_fcst`: colour codes used to represent the categorical bands in the forecast.


In [ ]:
# Select station to plot (CatchmentID)
station_ID = '1'

# Folder where files were saved
folder_output = 'output_hydrosos'

# Select forecast aggregation time
# Type: 'accumulated' (e.g., X-month averaged) or 'single' (for monthly lead time)
forecast_type = 'single'

# Color codes (HEX) according to HydroSOS standards (5 categories)
color_codes_fcst = ['#CD233f','#FFA885','#E7E2BC','#8ECEEE','#2C7DCD']

2.2. Loading historical status information outputs:

In [ ]:
# Time series (obsSim)
df_status = pd.read_csv(f'{folder_output}/status/status/{station_ID}_status.csv')
# Percentile bands
df_status_bands = pd.read_csv(f'{folder_output}/status/statusBands/{station_ID}_bands.csv')

print('Historical data:')
display(df_status)
print('Bands calculated using historical information:')
display(df_status_bands)

2.3. Loading the future hydrological outlook results:

In [ ]:
# Reading the output files from script forecastcalc.py
df_forecasts = pd.read_csv(f'{folder_output}/{forecast_type}/forecasts/{station_ID}_forecasts.csv')
df_counts = pd.read_csv(f'{folder_output}/{forecast_type}/counts/{station_ID}_counts.csv')
df_fcst_bands = pd.read_csv(f'{folder_output}/{forecast_type}/forecastBands/{station_ID}_bands.csv')
df_percentiles = pd.read_csv(f'{folder_output}/{forecast_type}/percentiles/{station_ID}_percentiles.csv')

# Setting indices to dataframes
df_forecasts.set_index('date', inplace=True)
df_counts.set_index('date',inplace=True)
df_fcst_bands.set_index('relative_month', inplace=True)

Next, we will work with these tables to visualize some results.

---
# **3. Verifying the hydrological outlook plots**

## 3.1. Forecast plots (ensemble traces)

The table in the "**forecasts**" folder contains the streamflow forecasts for each member, with monthly time step:

In [ ]:
display(df_forecasts)

In [ ]:
#@title Plot the graphs:
df_forecasts.plot(figsize=(12, 6),
                  legend=False,
                  color='gray',
                  ylabel='Streamflow (m³/s)',
                  xlabel='Date of forecast validity',
                  title=f'Ensemble traces({forecast_type})')

## 3.2. Number of members in each category (counts)

The table generated in the "**counts**" folder contains the number of ensemble members within each predefined flow category:

In [ ]:
display(df_counts)

We can obtain the ensemble size to calculate the % of members falling into each of the categories:

In [ ]:
# Calculating the percentage of members in each category
n_members = df_counts.sum(axis=1).values[0]
df_counts_prob = 100 * (df_counts / n_members)

display(df_counts_prob)

In [ ]:
#@title Plotting the forecast probabilities in bar charts
fig, ax = plt.subplots(figsize=(8, 6))

ax = df_counts_prob.plot(ax=ax, kind='bar', ylabel='Probability (%)',
                         xlabel='Forecast validity date', edgecolor='k',
                         color=color_codes_fcst, width=0.8, stacked='True')

# Set date labels to x axis
ax.set_xticklabels(df_counts_prob.index, rotation=45)

# Add the legend above the plot
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.set_title(f'% of members on each category ({forecast_type})')

plt.tight_layout()

## 3.3. Forecast category bands (forecast bands)

The table generated in the "**ForecastBands**" folder contains the percentiles (and other statistics) calculated from the historical daily data. These percentiles are relative to the month of the first date in our forecast (`relative_month = 1`):

In [ ]:
df_fcst_bands

Let's separate the statistics from the table to plot the results:

In [ ]:
# Get only minimum and maximum values of the reference period, in the corresponding months in the forecast period
df_fcst_bands_min_max = df_fcst_bands[['min','max']]

# Get only percentiles of the reference period (i.e., by excluding other statistics)
df_fcst_bands_prct = df_fcst_bands.drop(columns=['min','mean','max'])

In [ ]:
# @title Plotting the percentile bands
fig,ax = plt.subplots(figsize=(8, 6))

##################################################3
# First part: Ploting the percentile bands

x = df_fcst_bands_prct.index  # Relative months starting from the forecast issue date
percentile_cols = df_fcst_bands_prct.columns # Names of the percentile columns (e.g., '10%', '25%', etc.)
n_bands = len(percentile_cols) - 1 # Number of bands between percentiles

# Band below the lowest climatological threshold (from 0 to first percentile)
ax.fill_between(x, 0, df_fcst_bands_prct.iloc[:, 0],
                 color=color_codes_fcst[0], alpha=0.7, label=f'<{percentile_cols[0]}'
)

# Bands between climatological percentiles
for i in range(n_bands):
    ax.fill_between(x, df_fcst_bands_prct.iloc[:, i], df_fcst_bands_prct.iloc[:, i + 1],
        color=color_codes_fcst[i + 1], label=f'{percentile_cols[i]}-{percentile_cols[i+1]}'
    )

# Band above the highest climatological threshold (from last percentile to max)
ax.fill_between(x, df_fcst_bands_prct.iloc[:, -1], df_fcst_bands_min_max['max'],
    color=color_codes_fcst[n_bands + 1], label=f'>{percentile_cols[-1]}'
)

# Plotting minimum and maximum historical (simulated) values
ax.plot(df_fcst_bands_min_max['min'], linestyle='--', color='silver', label='historical min')
ax.plot(df_fcst_bands_min_max['max'], linestyle='--', color='black', label='historical max')


##################################################
# Second part: Ploting boxplots for the forecast ensemble

# Numbers to match the x-axis of the boxplot with the dates
positions = np.arange(1,len(df_forecasts.index) + 1)

ax.boxplot(
    [df_forecasts.loc[date] for date in df_forecasts.index],
     positions=positions,
     widths=0.6,
     patch_artist=True,
     boxprops=dict(facecolor='grey', color='black', alpha=0.3),
     medianprops=dict(color='black'),
     whiskerprops=dict(color='black'),
     capprops=dict(color='black'),
     flierprops=dict(markerfacecolor='black', marker='o', markersize=3, alpha=0.5)
 )

ax.set_xticklabels(df_forecasts.index, rotation=45)

plt.xlabel('Forecast validity date')
plt.ylabel('Streamflow (m³/s)')
plt.title(f'ESP forecast ({forecast_type})')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.xlim(0.5, len(df_forecasts.index) + 0.5)
plt.tight_layout()
plt.show()

Finally, the table generated in the "percentiles" folder contains statistics for our ensemble:

In [ ]:
display(df_percentiles)

---
# **4. Test your knowledge**



Now you will be responsible for generating a HydroSOS hydrological outlook product. The instructions are as follows:

**1. Generating the forecast**

* Use **Notebook 4 - Ensemble Streamflow Prediction (ESP)** to generate an ESP forecast starting on **2025-01-01** with a lead time of 6 months (**180 days**).

**2. Saving the .csv files**

* Save the forecast results in **individual .csv files for each member**. The file names must be compatible with the *forecast_dir* example data used by the `forecastcalc.py` routine.

* Save the .csv file of the **historical streamflow simulated** by the hydrological model, filtered for the period between **1991-01-01 and 2020-12-31** (these will be used instead of the observed data). The file name must be compatible with the *obs_dir* example data used by the `forecastcalc.py` routine.

**Suggestion**: Use the *shell* command to compress the ESP .csv files into a .zip file to make them easier to handle in Colab.

**3. Generating the HydroSOS tables with the hydrological outlooks**

* Using this notebook, run the `forecastcalc.py` routine with the files generated in step 2.

**4. Plotting the result**

* Generate the bar chart with the ESP probabilities (single aggregation).

* Submit the file (figure) with the result to the link indicated (Section **4 - HydroSOS Methodology -> Task submission**).

Note: If for any reason the chart could not be generated correctly, you can also submit the table with the calculated probabilities for each category (df_counts_prob).